In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Unzip model
import zipfile, os

MODEL_ZIP  = "/content/drive/MyDrive/arrowhead-model/wav2vec2-shemo-bfsi.zip"
MODEL_PATH = "/content/wav2vec2-shemo-bfsi"

with zipfile.ZipFile(MODEL_ZIP, 'r') as z:
    z.extractall(MODEL_PATH)

print("Model extracted to:", MODEL_PATH)
print("Files:", os.listdir(MODEL_PATH))

Mounted at /content/drive
Model extracted to: /content/wav2vec2-shemo-bfsi
Files: ['kaggle']


In [2]:
!pip install -q transformers torchaudio librosa
!pip install -q openai-whisper
!pip install -q TTS
!pip install -q bitsandbytes accelerate
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.

Python Version

In [3]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [4]:
!pip install -q edge-tts gradio openai-whisper transformers torchaudio librosa accelerate bitsandbytes

In [5]:
!pip install -q nest_asyncio

In [6]:
import edge_tts
import asyncio
import nest_asyncio
import tempfile

# Fix for Colab's existing event loop
nest_asyncio.apply()

async def _tts_async(text: str, output_path: str):
    communicate = edge_tts.Communicate(
        text=text,
        voice="en-IN-NeerjaNeural",
    )
    await communicate.save(output_path)

def text_to_speech(text: str, output_path: str = None) -> str:
    if output_path is None:
        output_path = tempfile.mktemp(suffix=".wav")
    loop = asyncio.get_event_loop()
    loop.run_until_complete(_tts_async(text, output_path))
    return output_path

# Test it
test_audio = text_to_speech("Hello, I understand your concern. Let me help you with your account.")
print(f"TTS working. Output: {test_audio}")

from IPython.display import Audio, display
display(Audio(test_audio))

TTS working. Output: /tmp/tmpb52uc354.wav


In [7]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification
import torch
import torchaudio
import numpy as np

MODEL_PATH = "/content/wav2vec2-shemo-bfsi/kaggle/working/wav2vec2-shemo-bfsi"  # where you extracted the zip

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_PATH)
emotion_model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_PATH)
emotion_model.eval()
emotion_model.cuda()

ID2LABEL    = {0: "calm", 1: "frustrated", 2: "disengaged"}
TARGET_SR   = 16000
MAX_SAMPLES = TARGET_SR * 6

def predict_emotion(audio_path: str):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
    audio = waveform.squeeze().numpy()
    if len(audio) > MAX_SAMPLES:
        audio = audio[:MAX_SAMPLES]
    else:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
    inputs = feature_extractor(
        audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True,
    )
    with torch.no_grad():
        logits = emotion_model(inputs.input_values.cuda()).logits
    probs           = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    predicted_id    = int(np.argmax(probs))
    predicted_label = ID2LABEL[predicted_id]
    confidence      = float(probs[predicted_id])
    return {
        "emotion":    predicted_label,
        "confidence": round(confidence, 4),
        "scores": {
            "calm":       round(float(probs[0]), 4),
            "frustrated": round(float(probs[1]), 4),
            "disengaged": round(float(probs[2]), 4),
        }
    }

print("Emotion model loaded")

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Emotion model loaded


In [8]:
import whisper

print("Loading Whisper large-v3... (takes ~2 mins)")
whisper_model = whisper.load_model("large-v3")
print("Whisper loaded")

Loading Whisper large-v3... (takes ~2 mins)


100%|█████████████████████████████████████| 2.88G/2.88G [00:41<00:00, 74.2MiB/s]


Whisper loaded


Memory Check

In [9]:
# import torch
# for i in range(torch.cuda.device_count()):
#     total = torch.cuda.get_device_properties(i).total_memory / 1e9
#     used  = torch.cuda.memory_allocated(i) / 1e9
#     free  = total - used
#     print(f"GPU {i}: {total:.1f}GB total | {used:.1f}GB used | {free:.1f}GB free")

Findin where Exctly model is

In [10]:
# import os

# # Find where the model files actually are
# for root, dirs, files in os.walk("/content"):
#     for f in files:
#         if f == "config.json":
#             print(os.path.join(root, f))

Clear Memory...if overloaded on GPU

In [11]:
# import gc
# import torch

# # Delete everything
# try: del llama_model
# except: pass
# try: del llama_tokenizer
# except: pass
# try: del whisper_model
# except: pass
# try: del emotion_model
# except: pass

# torch.cuda.empty_cache()
# gc.collect()
# torch.cuda.empty_cache()

# for i in range(torch.cuda.device_count()):
#     total = torch.cuda.get_device_properties(i).total_memory / 1e9
#     used  = torch.cuda.memory_allocated(i) / 1e9
#     free  = total - used
#     print(f"GPU {i}: {total:.1f}GB total | {used:.1f}GB used | {free:.1f}GB free")

In [12]:
def make_escalation_detector():
    buffer    = []
    WINDOW    = 3
    THRESHOLD = 0.60
    DISTRESS  = ["frustrated", "disengaged"]

    def check(audio_path: str):
        result      = predict_emotion(audio_path)
        emotion     = result["emotion"]
        confidence  = result["confidence"]
        is_distress = emotion in DISTRESS and confidence > THRESHOLD
        buffer.append(is_distress)
        if len(buffer) > WINDOW:
            buffer.pop(0)
        escalate = len(buffer) == WINDOW and all(buffer)
        return {
            "emotion":    emotion,
            "confidence": confidence,
            "scores":     result["scores"],
            "escalate":   escalate,
            "reason":     f"sustained_{emotion}" if escalate else None,
            "buffer":     list(buffer),
        }
    return check

detector = make_escalation_detector()
print("Escalation detector ready")

Escalation detector ready


In [ ]:
import gc, torch

try: del llama_model
except: pass
try: del llama_tokenizer
except: pass
torch.cuda.empty_cache()
gc.collect()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN    = "ENTER YOUR TOKEN HERE"   # ← Enter your hugging face token here
LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading Llama-3.2-3B in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

llama_tokenizer = AutoTokenizer.from_pretrained(
    LLAMA_MODEL, token=HF_TOKEN
)

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)

llama_model.eval()
print("Llama-3.2-3B loaded successfully")

for i in range(torch.cuda.device_count()):
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    used  = torch.cuda.memory_allocated(i) / 1e9
    free  = total - used
    print(f"GPU {i}: {total:.1f}GB total | {used:.1f}GB used | {free:.1f}GB free")

Loading Llama-3.2-3B in 4-bit...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Llama-3.2-3B loaded successfully
GPU 0: 15.6GB total | 8.9GB used | 6.7GB free


In [14]:
def generate_response(transcript: str, emotion: str, history: list) -> str:
    system_prompt = f"""You are an empathetic AI assistant for Arrowhead Finance, a debt collection company.
The borrower's current emotional state is: {emotion.upper()}

Respond based on emotion:
- calm: Be professional and direct. Offer repayment options clearly.
- frustrated: Acknowledge frustration first. Be empathetic before discussing payment.
- disengaged: Ask a short open question to re-engage. Keep response under 2 sentences.

Always be respectful. Never be aggressive. Keep response under 3 sentences."""

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history)
    messages.append({"role": "user", "content": transcript})

    input_text = llama_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llama_tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = llama_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7,
            do_sample=True,
            pad_token_id=llama_tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response   = llama_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response

print("LLM response function ready")

LLM response function ready


In [15]:
def run_pipeline(audio_path: str, history: list):
    # STT
    transcript = whisper_model.transcribe(audio_path)["text"].strip()

    # Emotion
    emotion_result  = predict_emotion(audio_path)
    emotion         = emotion_result["emotion"]
    confidence      = emotion_result["confidence"]
    scores          = emotion_result["scores"]

    # Escalation
    escalation_result = detector(audio_path)
    should_escalate   = escalation_result["escalate"]

    # LLM response
    if should_escalate:
        bot_text = "I understand this has been difficult. Let me connect you with a senior advisor who can help you right now."
    else:
        bot_text = generate_response(transcript, emotion, history)

    # TTS
    audio_output = text_to_speech(bot_text)

    # Update history
    history.append({"role": "user",      "content": transcript})
    history.append({"role": "assistant", "content": bot_text})

    return transcript, emotion, confidence, scores, should_escalate, bot_text, audio_output, history

print("Full pipeline ready")

Full pipeline ready


In [16]:
# Download a sample audio file for testing
!wget -q "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac" -O /content/test_audio.wav
test_wav = "/content/test_audio.wav"
print("Test audio ready")

Test audio ready


In [17]:
display(Audio("/content/test_audio.wav"))

In [18]:
transcript, emotion, confidence, scores, escalate, bot_text, audio_out, history = \
    run_pipeline(test_wav, [])

print(f"Transcript  : {transcript}")
print(f"Emotion     : {emotion} ({confidence:.2f})")
print(f"Escalate    : {escalate}")
print(f"Bot response: {bot_text}")

from IPython.display import Audio, display
display(Audio(audio_out))

Transcript  : I have a dream that one day this nation will rise up and live out the true meaning of its creed.
Emotion     : frustrated (0.89)
Escalate    : False
Bot response: I sense that you're feeling overwhelmed and stressed about your debt, and it's taking a toll on your mind. Can you tell me a bit more about what's been going on with your debt that's causing you so much frustration?


In [24]:
!pip install -q streamlit
!npm install -g localtunnel 2>/dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.4 MB/s eta 0:00:00

added 22 packages in 2s

3 packages are looking for funding
  run `npm fund` for details


In [25]:
app_code = '''
import streamlit as st
import torch
import torchaudio
import numpy as np
import tempfile
import asyncio
import edge_tts
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import whisper

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title="Arrowhead BFSI Voice Assistant",
    page_icon="🎙️",
    layout="wide"
)

st.title("🎙️ Arrowhead BFSI Voice Emotion Assistant")
st.markdown("**Real-time emotion-aware debt collection call assistant**")
st.divider()

# ── Load models (cached so they load once) ───────────────────
@st.cache_resource
def load_models():
    MODEL_PATH = "/content/wav2vec2-shemo-bfsi/kaggle/working/wav2vec2-shemo-bfsi"

    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_PATH)
    emotion_model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_PATH)
    emotion_model.eval()
    emotion_model.cuda()

    whisper_model = whisper.load_model("small")

    HF_TOKEN    = "your_hf_token_here"   # ← replace
    LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
    bnb_config  = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL, token=HF_TOKEN)
    llama_model     = AutoModelForCausalLM.from_pretrained(
        LLAMA_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    )
    llama_model.eval()

    return feature_extractor, emotion_model, whisper_model, llama_tokenizer, llama_model

feature_extractor, emotion_model, whisper_model, llama_tokenizer, llama_model = load_models()

ID2LABEL    = {0: "calm", 1: "frustrated", 2: "disengaged"}
TARGET_SR   = 16000
MAX_SAMPLES = TARGET_SR * 6

# ── Helper functions ─────────────────────────────────────────
def predict_emotion(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
    audio = waveform.squeeze().numpy()
    audio = audio[:MAX_SAMPLES] if len(audio) > MAX_SAMPLES else np.pad(audio, (0, MAX_SAMPLES - len(audio)))
    inputs = feature_extractor(audio, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = emotion_model(inputs.input_values.cuda()).logits
    probs      = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id    = int(np.argmax(probs))
    return {
        "emotion":    ID2LABEL[pred_id],
        "confidence": round(float(probs[pred_id]), 4),
        "scores":     {ID2LABEL[i]: round(float(probs[i]), 4) for i in range(3)},
    }

def generate_response(transcript, emotion, history):
    system_prompt = f"""You are an empathetic AI assistant for Arrowhead Finance debt collection.
Borrower emotion: {emotion.upper()}
- calm: Be professional, offer repayment options.
- frustrated: Acknowledge frustration first, then discuss payment.
- disengaged: Ask a short open question to re-engage.
Keep response under 3 sentences. Be respectful always."""
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history)
    messages.append({"role": "user", "content": transcript})
    input_text = llama_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs     = llama_tokenizer(input_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = llama_model.generate(
            **inputs, max_new_tokens=120, temperature=0.7,
            do_sample=True, pad_token_id=llama_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return llama_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

async def _tts(text, path):
    await edge_tts.Communicate(text=text, voice="en-IN-NeerjaNeural").save(path)

def text_to_speech(text):
    path = tempfile.mktemp(suffix=".wav")
    asyncio.get_event_loop().run_until_complete(_tts(text, path))
    return path

# ── Session state ─────────────────────────────────────────────
if "history"       not in st.session_state: st.session_state.history       = []
if "distress_buf"  not in st.session_state: st.session_state.distress_buf  = []
if "chat_log"      not in st.session_state: st.session_state.chat_log      = []

# ── UI ────────────────────────────────────────────────────────
col1, col2 = st.columns(2)

with col1:
    st.subheader("📤 Input")
    uploaded = st.file_uploader("Upload audio (.wav, .mp3)", type=["wav", "mp3"])

    if st.button("🔍 Analyze & Respond", type="primary", disabled=uploaded is None):
        with st.spinner("Processing..."):
            # Save uploaded file
            tmp = tempfile.mktemp(suffix=".wav")
            with open(tmp, "wb") as f:
                f.write(uploaded.read())

            # STT
            transcript = whisper_model.transcribe(tmp)["text"].strip()

            # Emotion
            result     = predict_emotion(tmp)
            emotion    = result["emotion"]
            confidence = result["confidence"]
            scores     = result["scores"]

            # Escalation
            is_distress = emotion in ["frustrated", "disengaged"] and confidence > 0.60
            st.session_state.distress_buf.append(is_distress)
            if len(st.session_state.distress_buf) > 3:
                st.session_state.distress_buf.pop(0)
            escalate = len(st.session_state.distress_buf) == 3 and all(st.session_state.distress_buf)

            # LLM
            if escalate:
                bot_text = "I understand this has been difficult. Let me connect you with a senior advisor who can help you right now."
            else:
                bot_text = generate_response(transcript, emotion, st.session_state.history)

            # TTS
            audio_path = text_to_speech(bot_text)

            # Update history
            st.session_state.history.append({"role": "user",      "content": transcript})
            st.session_state.history.append({"role": "assistant",  "content": bot_text})
            st.session_state.chat_log.append({
                "transcript": transcript, "emotion": emotion,
                "confidence": confidence, "scores": scores,
                "escalate": escalate, "bot_text": bot_text,
                "audio_path": audio_path,
            })

    if st.button("🗑️ Clear conversation"):
        st.session_state.history      = []
        st.session_state.distress_buf = []
        st.session_state.chat_log     = []
        st.rerun()

with col2:
    st.subheader("📊 Analysis")
    if st.session_state.chat_log:
        latest = st.session_state.chat_log[-1]

        # Emotion badge
        color_map = {"calm": "🟢", "frustrated": "🟠", "disengaged": "⚪"}
        icon = color_map.get(latest["emotion"], "🔵")
        st.metric("Detected Emotion", f"{icon} {latest['emotion'].upper()}", f"{latest['confidence']*100:.1f}% confidence")

        # Scores
        st.markdown("**Emotion scores:**")
        for label, score in latest["scores"].items():
            st.progress(score, text=f"{label}: {score*100:.1f}%")

        # Escalation
        if latest["escalate"]:
            st.error("🚨 ESCALATE — Transfer to human agent now")
        else:
            st.success("✅ OK — Bot can continue")

        st.divider()
        st.markdown(f"**📝 Transcript:** {latest['transcript']}")
        st.markdown(f"**🤖 Bot response:** {latest['bot_text']}")
        st.audio(latest["audio_path"])

    else:
        st.info("Upload an audio file and click Analyze to see results here.")

# ── Conversation history ──────────────────────────────────────
if st.session_state.chat_log:
    st.divider()
    st.subheader("💬 Conversation History")
    for i, turn in enumerate(st.session_state.chat_log):
        with st.expander(f"Turn {i+1} — {turn['emotion'].upper()} ({turn['confidence']*100:.1f}%)"):
            st.write(f"**You:** {turn['transcript']}")
            st.write(f"**Bot:** {turn['bot_text']}")
            if turn['escalate']:
                st.error("🚨 Escalation triggered this turn")
'''

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("app.py written successfully")

app.py written successfully


StreamLit Demo

In [26]:
import subprocess, threading, time

def run_streamlit():
    subprocess.run(["streamlit", "run", "/content/app.py",
                    "--server.port", "8501",
                    "--server.headless", "true",
                    "--server.enableCORS", "false"])

# Start streamlit in background
t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(5)

# Create public tunnel
print("Starting tunnel...")
!npx localtunnel --port 8501 &
time.sleep(3)
print("Your app is live at the URL shown above ↑")
print("(look for 'your url is: https://xxxxx.loca.lt')")

Starting tunnel...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦your url is: https://dirty-maps-greet.loca.lt
Your app is live at the URL shown above ↑
(look for 'your url is: https://xxxxx.loca.lt')


Demo in Colab UI only.....this will require less ram and will directly use the model already trained above...instead of loading it from scratch like streamlit

In [27]:
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import tempfile, os

# Upload button
upload = widgets.FileUpload(accept='.wav,.mp3', multiple=False, description='Upload Audio')
analyze_btn = widgets.Button(description='🔍 Analyze & Respond', button_style='primary')
clear_btn = widgets.Button(description='🗑️ Clear', button_style='warning')
output_area = widgets.Output()

conversation_history = []
distress_buffer = []

def on_analyze(b):
    with output_area:
        clear_output()
        if not upload.value:
            print("Please upload an audio file first.")
            return

        # Save uploaded file
        uploaded_file = list(upload.value.values())[0]
        tmp = tempfile.mktemp(suffix=".wav")
        with open(tmp, 'wb') as f:
            f.write(uploaded_file['content'])

        print("⏳ Processing...")

        # STT
        transcript = whisper_model.transcribe(tmp)["text"].strip()

        # Emotion
        result     = predict_emotion(tmp)
        emotion    = result["emotion"]
        confidence = result["confidence"]
        scores     = result["scores"]

        # Escalation
        is_distress = emotion in ["frustrated", "disengaged"] and confidence > 0.60
        distress_buffer.append(is_distress)
        if len(distress_buffer) > 3:
            distress_buffer.pop(0)
        escalate = len(distress_buffer) == 3 and all(distress_buffer)

        # LLM
        if escalate:
            bot_text = "I understand this has been difficult. Let me connect you with a senior advisor who can help you right now."
        else:
            bot_text = generate_response(transcript, emotion, conversation_history)

        # TTS
        audio_path = text_to_speech(bot_text)

        # Update history
        conversation_history.append({"role": "user",      "content": transcript})
        conversation_history.append({"role": "assistant", "content": bot_text})

        # Display results
        print("=" * 55)
        print(f"📝 Transcript   : {transcript}")
        print(f"🎭 Emotion      : {emotion.upper()} ({confidence*100:.1f}% confidence)")
        print(f"📊 Scores       : calm={scores['calm']*100:.0f}%  frustrated={scores['frustrated']*100:.0f}%  disengaged={scores['disengaged']*100:.0f}%")
        if escalate:
            print(f"🚨 Escalation   : ESCALATE — Transfer to human agent now")
        else:
            print(f"✅ Escalation   : OK — Bot can continue")
        print(f"🤖 Bot response : {bot_text}")
        print("=" * 55)

        # Play bot audio
        display(Audio(audio_path, autoplay=True))

def on_clear(b):
    conversation_history.clear()
    distress_buffer.clear()
    with output_area:
        clear_output()
    print("Conversation cleared.")

analyze_btn.on_click(on_analyze)
clear_btn.on_click(on_clear)

print("🎙️ Arrowhead BFSI Voice Emotion Assistant")
print("─" * 45)
print("Upload a .wav file and click Analyze & Respond")
print()
display(upload, widgets.HBox([analyze_btn, clear_btn]), output_area)

🎙️ Arrowhead BFSI Voice Emotion Assistant
─────────────────────────────────────────────
Upload a .wav file and click Analyze & Respond



FileUpload(value={}, accept='.wav,.mp3', description='Upload Audio')

Output()

1. Test escalation trigger — upload the same file 3 times in a row and confirm.. ESCALATE fires on the 3rd click.